# The Ultimate Hyperparameter Tuning Cheat Sheet

There are dozens of tuning libraries in Python, but you only need to remember these three to handle 99% of Machine Learning and Deep Learning tasks.


## 1. `RandomizedSearchCV` (The Everyday Workhorse)
Built directly into Scikit-Learn. Instead of testing every single combination (like GridSearch), it tests a random sample of parameters, saving massive amounts of time while getting almost identical results.

* **When to use it:** When you are training standard, classic Machine Learning models.
* **Where to use it:** With Scikit-Learn models like `RandomForest`, `SVC`, or `LogisticRegression`.

**Pros:**
* **Zero Setup:** Already built into `sklearn`. No extra libraries to install.
* **Fast:** Finds a "good enough" answer in a fraction of the time of GridSearch.
* **Simple:** The code is very short and easy to read.

**Cons:**
* **Not Smart:** It relies purely on luck. It does not learn from its previous mistakes.
* **No Deep Learning:** Cannot easily dynamically change neural network architectures (like adding or removing layers).

```python

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the parameter grid
param_dist = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# 2. Set up the search
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(),
    param_distributions=param_dist,
    n_iter=10,          # Pick 10 random combinations to try
    cv=3,               # 3-Fold Cross Validation
    scoring='accuracy'
)

# 3. Train and get best
random_search.fit(X_train, y_train)
print("Best Parameters:", random_search.best_params_)


```


## 2. `KerasTuner` (The Deep Learning Easy Button)
The official hyperparameter tuner built by the TensorFlow/Keras team. It is designed specifically to optimize Artificial Neural Networks (ANNs).

* **When to use it:** When you are building Deep Learning models and need to figure out how many layers, neurons, or dropout rates to use.
* **Where to use it:** Strictly within the TensorFlow/Keras ecosystem. Use the `kt.BayesianOptimization` search algorithm for the best results.

**Pros:**
* **Architecture Tuning:** It can dynamically add or remove entire hidden layers during the search.
* **Native Integration:** Uses the exact same `model.compile` and `model.fit` syntax you already know.
* **Smart Search:** Bayesian Optimization learns from past trials to guess the best parameters.

**Cons:**
* **One Trick Pony:** Only works for Keras/TensorFlow. You cannot use it to tune a Random Forest or XGBoost model.
* **Can be slow:** If not configured with Early Stopping, Deep Learning tuning takes a very long time.

```python

import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

# 1. Define the model building function
def build_model(hp):
    model = keras.Sequential()
    
    # Let tuner decide number of neurons
    neurons = hp.Int('units', min_value=32, max_value=128, step=32)
    model.add(layers.Dense(units=neurons, activation='relu'))
    model.add(layers.Dense(1, activation='linear'))
    
    # Let tuner decide learning rate
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss='mse')
    
    return model

# 2. Set up the search
tuner = kt.BayesianOptimization(
    build_model,
    objective='val_loss',
    max_trials=10,
    directory='tuning_dir',
    project_name='keras_tune'
)

# 3. Train and get best
tuner.search(X_train, y_train, epochs=20, validation_split=0.2)
best_model = tuner.get_best_models(num_models=1)[0]


```

## 3. `Optuna` (The Ultimate Kaggle Weapon)
The professional Gold Standard for hyperparameter tuning. It uses advanced Bayesian statistics (TPE) to find the absolute best model in the shortest amount of time.

* **When to use it:** When you are competing on Kaggle, working on a professional production model, or need maximum performance.
* **Where to use it:** With **anything**. XGBoost, LightGBM, PyTorch, Keras, or Scikit-Learn. 

**Pros:**
* **Aggressive Pruning:** It ruthlessly monitors models as they train. If a model performs poorly on epoch 3, Optuna kills it immediately to save time.
* **Universal:** You only need to learn its syntax once, and you can use it on any algorithm in existence.
* **Incredible Visualizations:** Comes with built-in plots to show you exactly which hyperparameters impacted your model the most.

**Cons:**
* **Slight Learning Curve:** You have to write a custom `objective(trial)` function, which takes a few minutes to learn.
* **Overkill:** It is a heavy-duty tool. Using it for a very simple Logistic Regression model is like using a sledgehammer to crack a nut.

```python
import optuna
import xgboost as xgb
from sklearn.metrics import accuracy_score

# 1. Define the objective function
def objective(trial):
    # Let Optuna suggest parameters
    param = {
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300)
    }
    
    # Build and train model
    model = xgb.XGBClassifier(**param)
    model.fit(X_train, y_train)
    
    # Return the score to Optuna
    preds = model.predict(X_val)
    return accuracy_score(y_val, preds)

# 2. Set up and run the study
study = optuna.create_study(direction='maximize') # maximize because it's accuracy
study.optimize(objective, n_trials=20)

# 3. Get best results
print("Best Parameters:", study.best_params)



---

## Optuna for classification

In [ ]:
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from optuna.integration import TFKerasPruningCallback

# --- SET YOUR NUMBER OF CLASSES HERE ---
NUM_CLASSES = 10 # Example: 10 for MNIST, 3 for Sentiment Analysis

def objective_classification(trial):
    # 1. Clear Keras memory session
    keras.backend.clear_session()
    
    # 2. Get the number of input features (columns) from X_train
    num_features = X_train.shape[1] 

    # 3. Suggest hyperparameters to try
    n_layers = trial.suggest_int('n_layers', 1, 3)
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    
    model = keras.Sequential()
    
    # 4. Build the hidden layers dynamically
    for i in range(n_layers):
        num_neurons = trial.suggest_int(f'n_units_l{i}', 32, 128, step=32)
        
        # Feed input_dim ONLY into the very first dense layer
        if i == 0:
            model.add(layers.Dense(num_neurons, activation='relu', input_dim=num_features))
        else:
            model.add(layers.Dense(num_neurons, activation='relu'))
        
        dropout_rate = trial.suggest_float(f'dropout_l{i}', 0.1, 0.4, step=0.1)
        model.add(layers.Dropout(dropout_rate))
        
    # 5. Output layer (Softmax for multiclass classification)
    model.add(layers.Dense(NUM_CLASSES, activation='softmax'))
    
    # 6. Compile the model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='sparse_categorical_crossentropy', # Use 'categorical_crossentropy' if labels are One-Hot encoded
        metrics=['accuracy']
    )
    
    # 7. Fit the model with the Optuna Pruning Callback
    history = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=32,
        verbose=0, # Keeps output clean, preventing Jupyter notebook lag
        callbacks=[TFKerasPruningCallback(trial, 'val_accuracy')]
    )
    
    # Save the Model and History directly into Optuna's memory
    trial.set_user_attr('model', model)
    trial.set_user_attr('history', history.history)

    # 8. Return the best validation accuracy achieved during training
    return max(history.history['val_accuracy'])


## Optuna for regression

In [ ]:
def build_model(trial):
    # clears all the previous models from the memory, saving the gpu from being crash
    keras.backend.clear_session()

    # here optuna will choose the number of parameters btwn 1 to 5
    n_layers = trial.suggest_int('n_layes', 1,5)

    # here optuna will choose the learning rate
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    model = keras.Sequential()

    # takes total nums of cols in x_train
    cols = x_train.shape[1]

    # ---------------------------------------------------------------------------
    # adding layers and neurons
    # ---------------------------------------------------------------------------
    for i in range(n_layers):
        # chooses a random number of neuron for every layer
        num_neurons = trial.suggest_int(f"layer_{i}", 32, 128, step=32)

        if i == 0:
            # putting input_dim
            model.add(layers.Dense(num_neurons, activation='relu', input_dim=cols))
        else:
            model.add(layers.Dense(num_neurons, activation='relu'))

        # putting dropout
        droput_percentage = trial.suggest_float(f"dropout_{i}", 0.1,0.4, step=0.1)
        model.add(layers.Dropout(droput_percentage))

    # final layer of the model
    model.add(layers.Dense(1, activation='linear'))


    # ---------------------------------------------------------------------------
    # compiling the model
    # ---------------------------------------------------------------------------    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',        # using mean squared error for loss
        metrics=['mae']
    )


    # ---------------------------------------------------------------------------
    # model.fit
    # ---------------------------------------------------------------------------
    history = model.fit(
        x_train, y_train,
        validation_split=0.3,
        # validation_data=(x_test, y_test),          # either use validation_split or validation_data, cant use both
        epochs=50,
        verbose=0,           # keeps the jupyter notebook clean
        batch_size=32,
        callbacks=[TFKerasPruningCallback(trial, 'val_loss')]
    )

    # ---------------------------------------------------------------------------
    # storing different values in object
    # ---------------------------------------------------------------------------
    mae = min(history.history['mae'])
    val_mae = min(history.history['val_mae'])

    loss = min(history.history['loss'])
    val_loss = min(history.history['val_loss'])

    # ---------------------------------------------------------------------------
    # storing the model and values to the RAM
    # ---------------------------------------------------------------------------

    trial.set_user_attr('model', model)
    trial.set_user_attr('history', history.history)
    trial.set_user_attr('mae', mae)
    trial.set_user_attr('val_mae', val_mae)
    trial.set_user_attr('loss', loss)
    trial.set_user_attr('val_loss', val_loss)
    
    # Return exactly ONE thing for Optuna to minimize
    # you cant return all the metric, so we just saved all the metric and returned only one value
    return val_loss


